In [7]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from typing import Literal

In [8]:
class TicketState(BaseModel):
    issue : str = ""
    category : str = ""
    response: str = ""

In [9]:
def classify(state:TicketState) -> TicketState:
    issue = state.issue.lower()
    if "payment" in issue or "amount" in issue:
        state.category = "billing"
    elif "refund" in issue or "cancel" in  issue:
        state.category = "refund"
    else:
        state.category = "status"


    return state

In [10]:
def handle_billing(state:TicketState) -> TicketState:
    print('Billing Related Issue')
    state.response = f"Issue Resolved for {state.category}"
    return state



def handle_refund(state:TicketState) -> TicketState:
    print('Refund Related Issue')
    state.response = f"Issue Resolved for {state.category}"
    return state


def handle_status(state:TicketState) -> TicketState:
    print('Status Related Issue')
    state.response = f"Issue Resolved for {state.category}"
    return state

In [11]:
def router_logic(state:TicketState) -> Literal["billing","refund","status"]:
    return state.category

In [12]:
graph = StateGraph(TicketState)
graph.add_node("classify", classify)
graph.add_node("handle_billing",handle_billing)
graph.add_node("handle_refund",handle_refund)
graph.add_node("handle_status",handle_status)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", router_logic),
{"billing":"handle_billing", "status":"handle_status", "refund":"handle_refund"}


graph.add_edge("handle_billing", END)
graph.add_edge("handle_refund", END)
graph.add_edge("handle_status", END)


graph = StateGraph(TicketState)
graph.add_node("classify", classify)
graph.add_node("handle_billing", handle_billing)
graph.add_node("handle_refund", handle_refund)
graph.add_node("handle_status", handle_status)

graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify",
    router_logic,
    {
        "billing": "handle_billing",
        "status": "handle_status",
        "refund": "handle_refund",
    },
)

graph.add_edge("handle_billing", END)
graph.add_edge("handle_refund", END)
graph.add_edge("handle_status", END)

final_graph = graph.compile()

In [13]:
res = final_graph.invoke({"issue": "I am waiting for my refund"})

Refund Related Issue


In [14]:
res

{'issue': 'I am waiting for my refund',
 'category': 'refund',
 'response': 'Issue Resolved for refund'}